# Research API Investigation — Phase 0

Test Brave Search, Brave Answers, Tavily, and SEC EDGAR APIs before building the Agentic Research layer. Each section is self-contained with **PASS** / **FAIL** output.

## How to run

| Mode | Steps |
|------|-------|
| **Full run** | Run all cells top-to-bottom (Kernel → Run All) |
| **Single section** | Run **0.1 Environment** first, then the section you want |

**Sections:** 0.1 Environment → 0.2 Brave Search → 0.2b Brave Answers → 0.3 Tavily Search → 0.4 A/B/C Comparison → 0.4b LLM Quality Eval → 0.4d Side-by-side View → 0.4e Hard Question Re-test → 0.5 SEC Ground-Truth Validation → 0.6 Deployment Decision Table → 0.7 Routing Policy Dry-Run

**Reference:** [docs/AGENTIC_RESEARCH.md](../docs/AGENTIC_RESEARCH.md)

---
## 0.1 Environment Setup

In [3]:
import os
import sys
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

print(f"Project root : {PROJECT_ROOT}")
print(f"Python       : {sys.version}")
print(f"Timestamp    : {datetime.utcnow().isoformat()}Z")

Project root : /home/deploy_invest_ai/investment-agent
Python       : 3.12.3 (main, Mar  3 2026, 12:15:18) [GCC 13.3.0]
Timestamp    : 2026-03-16T10:01:47.309932Z


/tmp/ipykernel_1068928/3448514284.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f"Timestamp    : {datetime.utcnow().isoformat()}Z")


---
## 0.2 Brave Search

Call Brave Search API with sample financial queries. Measure latency and snippet quality.

In [21]:
import time
import httpx

SAMPLE_QUERIES = [
    "AAPL FY2025 Q4: total revenue and YoY growth from official company release",
    "AAPL FY2025 Q4: Services revenue and YoY growth",
    "Apple SEC CIK and exact data.sec.gov submissions endpoint",
    "SEC annual vs quarterly filing forms for public companies",
    "SEC insider ownership reporting forms and deadlines",
    "SEC form for material events and filing deadline",
]

GROUND_TRUTH_QA = {
    "AAPL FY2025 Q4: total revenue and YoY growth from official company release": {
        "model_answer": "Apple reported FY2025 Q4 revenue of $102.5B, up 8% year-over-year.",
        "key_facts": ["$102.5B revenue", "8% YoY growth"],
        "sources": [
            "https://www.apple.com/newsroom/",
            "https://www.sec.gov/edgar/browse/?CIK=320193",
        ],
    },
    "AAPL FY2025 Q4: Services revenue and YoY growth": {
        "model_answer": "Apple Services revenue was $28.8B in FY2025 Q4, up 15% year-over-year.",
        "key_facts": ["$28.8B services revenue", "15% YoY growth"],
        "sources": [
            "https://www.apple.com/newsroom/",
            "https://www.sec.gov/edgar/browse/?CIK=320193",
        ],
    },
    "Apple SEC CIK and exact data.sec.gov submissions endpoint": {
        "model_answer": "Apple's CIK is 320193 (padded: 0000320193). Submissions endpoint: https://data.sec.gov/submissions/CIK0000320193.json",
        "key_facts": ["CIK 320193", "CIK0000320193.json endpoint"],
        "sources": [
            "https://www.sec.gov/files/company_tickers.json",
            "https://data.sec.gov/submissions/CIK0000320193.json",
        ],
    },
    "SEC annual vs quarterly filing forms for public companies": {
        "model_answer": "Form 10-K is the annual report; Form 10-Q is the quarterly report.",
        "key_facts": ["10-K annual", "10-Q quarterly"],
        "sources": [
            "https://www.sec.gov/forms",
        ],
    },
    "SEC insider ownership reporting forms and deadlines": {
        "model_answer": "Insider ownership reporting uses Forms 3, 4, and 5. Form 3 is initial ownership, Form 4 reports changes (typically within 2 business days), and Form 5 is annual catch-up for exempt/late transactions.",
        "key_facts": ["Forms 3/4/5", "Form 4 ~2 business days"],
        "sources": [
            "https://www.sec.gov/forms",
            "https://www.investor.gov/introduction-investing/investing-basics/glossary/form-4",
        ],
    },
    "SEC form for material events and filing deadline": {
        "model_answer": "Form 8-K is used to disclose material corporate events, generally due within 4 business days of the event.",
        "key_facts": ["Form 8-K", "4 business days"],
        "sources": [
            "https://www.sec.gov/forms",
        ],
    },
}

def call_brave_search(query: str, count: int = 5) -> dict:
    key = os.environ.get("BRAVE_SEARCH_API_KEY")
    if not key:
        return {"error": "BRAVE_SEARCH_API_KEY not set", "latency_ms": 0}
    url = "https://api.search.brave.com/res/v1/web/search"
    params = {"q": query, "count": count}
    headers = {"X-Subscription-Token": key}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, params=params, headers=headers, timeout=15)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("web", {}).get("results", [])
        snippets = [{"title": r.get("title", ""), "description": r.get("description", "")[:200]} for r in results[:5]]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

brave_results = []
for q in SAMPLE_QUERIES:
    r = call_brave_search(q)
    brave_results.append({"query": q, **r})
    status = "PASS" if r.get("ok") else "FAIL"
    print(f"[{status}] {q}: {r.get('latency_ms', 0):.0f}ms, {r.get('num_results', 0)} results")

brave_pass = all(r.get("ok") for r in brave_results)
print(f"\nSection 0.2: {'PASS' if brave_pass else 'FAIL'}")

[PASS] AAPL FY2025 Q4: total revenue and YoY growth from official company release: 954ms, 5 results
[PASS] AAPL FY2025 Q4: Services revenue and YoY growth: 928ms, 5 results
[PASS] Apple SEC CIK and exact data.sec.gov submissions endpoint: 977ms, 5 results
[PASS] SEC annual vs quarterly filing forms for public companies: 966ms, 5 results
[PASS] SEC insider ownership reporting forms and deadlines: 959ms, 5 results
[PASS] SEC form for material events and filing deadline: 958ms, 5 results

Section 0.2: PASS


In [5]:
brave_results

[{'query': 'AAPL FY2025 Q4: total revenue and YoY growth from official company release',
  'ok': True,
  'latency_ms': 1028.2622000668198,
  'num_results': 5,
  'snippets': [{'title': "Apple Q4 2025: Record $102.5B Revenue, Surging Services, and AI-Powered Future | The Acquirer's Multiple®",
    'description': '(NASDAQ: AAPL) closed fiscal 2025 with what CEO Tim Cook called “an extraordinary year for Apple, in which we achieved an all-time revenue record of $416 billion for the fiscal year.” Fourth-quarter r'},
   {'title': 'Apple Inc. (AAPL) Q4 FY2025 earnings call transcript',
    'description': 'Apple reported Q4 FY2025 revenue of <strong>$102.5B (+8% YoY),</strong> EPS of $1.85 (+13% YoY), and record services revenue of $28.8B (+15% YoY).'},
   {'title': 'Apple Revenue 2012-2025 | AAPL | MacroTrends',
    'description': 'Apple annual/quarterly revenue history and growth rate from 2012 to 2025. Revenue can be defined as the amount of money a company receives from its customers in ex

In [6]:
# Optional: inspect benchmark questions + ground truth
import pandas as pd

benchmark_rows = []
for q in SAMPLE_QUERIES:
    gt = GROUND_TRUTH_QA.get(q, {})
    benchmark_rows.append({
        "query": q,
        "model_answer": gt.get("model_answer", ""),
        "sources": " | ".join(gt.get("sources", [])),
    })

pd.set_option("display.max_colwidth", 220)
print(pd.DataFrame(benchmark_rows).to_string(index=False))

                                                                     query                                                                                                                                                                                            model_answer                                                                                                      sources
AAPL FY2025 Q4: total revenue and YoY growth from official company release                                                                                                                                      Apple reported FY2025 Q4 revenue of $102.5B, up 8% year-over-year.                               https://www.apple.com/newsroom/ | https://www.sec.gov/edgar/browse/?CIK=320193
                           AAPL FY2025 Q4: Services revenue and YoY growth                                                                                                                                  Apple Services revenue was $

---
## 0.2b Brave Answers

Call Brave Answers API (chat-completions style) with the same financial queries. Measure latency and answer quality signal.

In [22]:
def call_brave_answers(query: str, max_retries: int = 2) -> dict:
    key = os.environ.get("BRAVE_ANSWER_API_KEY")
    if not key:
        return {"error": "BRAVE_ANSWER_API_KEY not set", "latency_ms": 0}

    url = "https://api.search.brave.com/res/v1/chat/completions"
    headers = {
        "Content-Type": "application/json",
        "X-Subscription-Token": key.strip(),
    }
    body = {
        "model": "brave",
        "stream": False,
        "messages": [{"role": "user", "content": query}],
    }

    last_error = "unknown"
    total_latency_ms = 0.0

    for attempt in range(max_retries + 1):
        t0 = time.perf_counter()
        try:
            resp = httpx.post(url, json=body, headers=headers, timeout=45)
            latency_ms = (time.perf_counter() - t0) * 1000
            total_latency_ms += latency_ms

            if resp.status_code in (408, 429, 500, 502, 503, 504):
                last_error = f"HTTP {resp.status_code}: {resp.text[:180]}"
                if attempt < max_retries:
                    time.sleep(1.5 * (attempt + 1))
                    continue
                return {"error": last_error, "latency_ms": total_latency_ms}

            if resp.status_code != 200:
                return {"error": f"HTTP {resp.status_code}: {resp.text[:180]}", "latency_ms": total_latency_ms}

            data = resp.json()
            choices = data.get("choices", []) or []
            answer = ""
            if choices:
                answer = (((choices[0] or {}).get("message") or {}).get("content") or "")[:500]

            # Brave Answers may expose citations in provider-specific fields; keep this robust.
            citations = data.get("citations") or data.get("references") or []
            num_citations = len(citations) if isinstance(citations, list) else 0

            return {
                "ok": True,
                "latency_ms": total_latency_ms,
                "num_results": num_citations,
                "answer": answer,
                "snippets": [],
            }
        except Exception as e:
            latency_ms = (time.perf_counter() - t0) * 1000
            total_latency_ms += latency_ms
            last_error = str(e)
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue

    return {"error": last_error, "latency_ms": total_latency_ms}


brave_answers_results = []
for q in SAMPLE_QUERIES:
    r = call_brave_answers(q)
    brave_answers_results.append({"query": q, **r})
    status = "PASS" if r.get("ok") else "FAIL"
    msg = f"[{status}] {q}: {r.get('latency_ms', 0):.0f}ms, answer_len={len(r.get('answer', ''))}, citations={r.get('num_results', 0)}"
    if not r.get("ok"):
        msg += f", error={r.get('error', 'unknown')}"
    print(msg)

brave_answers_pass = all(r.get("ok") for r in brave_answers_results)
print(f"\nSection 0.2b: {'PASS' if brave_answers_pass else 'FAIL'}")

[PASS] AAPL FY2025 Q4: total revenue and YoY growth from official company release: 2821ms, answer_len=475, citations=0
[PASS] AAPL FY2025 Q4: Services revenue and YoY growth: 2258ms, answer_len=500, citations=0
[PASS] Apple SEC CIK and exact data.sec.gov submissions endpoint: 7326ms, answer_len=500, citations=0
[PASS] SEC annual vs quarterly filing forms for public companies: 5855ms, answer_len=500, citations=0
[PASS] SEC insider ownership reporting forms and deadlines: 19131ms, answer_len=500, citations=0
[PASS] SEC form for material events and filing deadline: 9613ms, answer_len=500, citations=0

Section 0.2b: PASS


In [14]:
brave_answers_results

[{'query': 'AAPL FY2025 Q4: total revenue and YoY growth from official company release',
  'ok': True,
  'latency_ms': 3197.697033174336,
  'num_results': 0,
  'answer': "The provided context does not include Apple Inc.'s total revenue or year-over-year growth for the fourth quarter of fiscal year 2025 from an official company release.\n\nApple Inc., a major player in the technology industry, designs, manufactures, and markets smartphones, tablets, computers, wearables, and related services. As of March 16, 2026, Apple's stock (AAPL) is trading at $250.12, with a market capitalization of approximately $3.68 trillion. The stock experienced a decline during the tradi",
  'snippets': []},
 {'query': 'AAPL FY2025 Q4: Services revenue and YoY growth',
  'ok': True,
  'latency_ms': 1995.3055090736598,
  'num_results': 0,
  'answer': "Apple Inc., a technology company operating in the consumer electronics, software, and digital services industries, has a market capitalization of approximately 

---
## 0.3 Tavily Search

Call Tavily Search API with `topic: finance`. Measure latency and content quality.

In [23]:
def call_tavily_search(query: str, max_results: int = 5) -> dict:
    key = os.environ.get("TAVILY_API_KEY")
    if not key:
        return {"error": "TAVILY_API_KEY not set", "latency_ms": 0}
    url = "https://api.tavily.com/search"
    headers = {"Content-Type": "application/json", "Authorization": f"Bearer {key.strip()}"}
    body = {"query": query, "search_depth": "basic", "topic": "finance", "max_results": max_results, "include_answer": True}
    t0 = time.perf_counter()
    try:
        resp = httpx.post(url, json=body, headers=headers, timeout=30)
        latency_ms = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency_ms}
        data = resp.json()
        results = data.get("results", [])
        answer = data.get("answer", "")[:300] if data.get("answer") else ""
        snippets = [{"title": r.get("title", ""), "content": (r.get("content", "") or "")[:200]} for r in results[:5]]
        return {"ok": True, "latency_ms": latency_ms, "num_results": len(results), "answer": answer, "snippets": snippets}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

tavily_results = []
for q in SAMPLE_QUERIES:
    r = call_tavily_search(q)
    tavily_results.append({"query": q, **r})
    status = "PASS" if r.get("ok") else "FAIL"
    print(f"[{status}] {q}: {r.get('latency_ms', 0):.0f}ms, {r.get('num_results', 0)} results")

tavily_pass = all(r.get("ok") for r in tavily_results)
print(f"\nSection 0.3: {'PASS' if tavily_pass else 'FAIL'}")

[PASS] AAPL FY2025 Q4: total revenue and YoY growth from official company release: 713ms, 5 results
[PASS] AAPL FY2025 Q4: Services revenue and YoY growth: 381ms, 5 results
[PASS] Apple SEC CIK and exact data.sec.gov submissions endpoint: 1914ms, 1 results
[PASS] SEC annual vs quarterly filing forms for public companies: 340ms, 5 results
[PASS] SEC insider ownership reporting forms and deadlines: 340ms, 5 results
[PASS] SEC form for material events and filing deadline: 343ms, 5 results

Section 0.3: PASS


In [16]:
tavily_results

[{'query': 'AAPL FY2025 Q4: total revenue and YoY growth from official company release',
  'ok': True,
  'latency_ms': 349.3760940618813,
  'num_results': 5,
  'answer': 'Apple Inc. reported total revenue of $102.5 billion for the fourth quarter of fiscal year 2025, marking an 8% year-over-year growth. This significant increase was driven by robust performance across various product lines, including the iPhone and Mac, as well as strong contributions from the company',
  'snippets': [{'title': 'Apple Inc. R (APC0.F) Q4 FY2025 earnings call transcript',
    'content': 'Apple reported Q4 FY2025 revenue of $102.5B (+8% YoY), EPS of $1.85'},
   {'title': 'Apple Inc. R (APC0.F) Q3 FY2025 earnings call transcript',
    'content': 'Products revenue was $66.6 billion, up 8% YoY, driven by growth across iPhone and Mac. Thanks to our high levels of customer satisfaction and'},
   {'title': "Apple's Desperate Pivot From Glass To Dopamine - Forbes",
    'content': '4.   Apple Pay prevented over $1

---
## 0.4 A/B/C Comparison

Side-by-side: same query -> Brave Search vs Brave Answers vs Tavily. Recommend primary vs fallback.

In [24]:
import pandas as pd

rows = []
for i, q in enumerate(SAMPLE_QUERIES):
    bs = brave_results[i] if i < len(brave_results) else {}
    ba = brave_answers_results[i] if i < len(brave_answers_results) else {}
    tv = tavily_results[i] if i < len(tavily_results) else {}
    rows.append({
        "query": q,
        "brave_search_latency_ms": bs.get("latency_ms", 0),
        "brave_search_ok": bs.get("ok", False),
        "brave_answers_latency_ms": ba.get("latency_ms", 0),
        "brave_answers_ok": ba.get("ok", False),
        "tavily_latency_ms": tv.get("latency_ms", 0),
        "tavily_ok": tv.get("ok", False),
        "brave_search_results": bs.get("num_results", 0),
        "brave_answers_citations": ba.get("num_results", 0),
        "tavily_results": tv.get("num_results", 0),
    })

df_abc = pd.DataFrame(rows)
print(df_abc.to_string())

avg_brave_search = df_abc["brave_search_latency_ms"].mean() if df_abc["brave_search_ok"].any() else 0
avg_brave_answers = df_abc["brave_answers_latency_ms"].mean() if df_abc["brave_answers_ok"].any() else 0
avg_tavily = df_abc["tavily_latency_ms"].mean() if df_abc["tavily_ok"].any() else 0

print(f"\nBrave Search avg latency: {avg_brave_search:.0f}ms")
print(f"Brave Answers avg latency: {avg_brave_answers:.0f}ms")
print(f"Tavily avg latency: {avg_tavily:.0f}ms")

if avg_brave_search and avg_brave_answers:
    primary = "Brave Search" if avg_brave_search <= avg_brave_answers else "Brave Answers"
    recommendation = f"{primary} primary, Tavily fallback"
else:
    recommendation = "Brave Search primary, Tavily fallback (run 0.2b to validate Brave Answers)"

print(f"\nRecommendation: {recommendation}")
print("\nSection 0.4: PASS (document recommendation in AGENTIC_RESEARCH.md)")

                                                                        query  brave_search_latency_ms  brave_search_ok  brave_answers_latency_ms  brave_answers_ok  tavily_latency_ms  tavily_ok  brave_search_results  brave_answers_citations  tavily_results
0  AAPL FY2025 Q4: total revenue and YoY growth from official company release               954.169886             True               2820.798393              True         713.483841       True                     5                        0               5
1                             AAPL FY2025 Q4: Services revenue and YoY growth               927.952365             True               2258.207848              True         381.103643       True                     5                        0               5
2                   Apple SEC CIK and exact data.sec.gov submissions endpoint               977.461098             True               7326.167105              True        1913.785493       True                     5              

In [18]:
# 0.4c Safety patch: prevent traceback on provider timeouts

def call_openai_quality_eval(evidence: list[dict], ground_truth: dict, model: str = "gpt-4o-mini") -> dict:
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        return {"error": "OPENAI_API_KEY not set"}

    prompt = (
        "You are evaluating search quality for an investment research agent. "
        "Use the provided ground truth to score each provider/query from 1-5 for: relevance, financial_signal, evidence_quality, noise_level (5 means very noisy), and truth_alignment (how well results align with ground truth key facts). "
        "Compute overall_quality as rounded average of relevance, financial_signal, evidence_quality, truth_alignment, and (6-noise_level). "
        "Return JSON only with schema including evaluations and provider_summary."
    )
    body = {
        "model": model,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": "Respond with valid JSON only."},
            {
                "role": "user",
                "content": (
                    prompt
                    + "\n\nGROUND_TRUTH:\n"
                    + json.dumps(ground_truth, ensure_ascii=False)
                    + "\n\nDATA:\n"
                    + json.dumps(evidence, ensure_ascii=False)
                ),
            },
        ],
    }

    try:
        resp = httpx.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json=body,
            timeout=45,
        )
    except httpx.TimeoutException as e:
        return {"error": f"OpenAI timeout: {e}"}
    except Exception as e:
        return {"error": f"OpenAI request failed: {e}"}

    if resp.status_code != 200:
        return {"error": f"OpenAI HTTP {resp.status_code}", "body": resp.text[:500]}

    try:
        return json.loads(resp.json()["choices"][0]["message"]["content"])
    except Exception:
        return {"raw": str(resp.text)[:2000]}


def call_anthropic_quality_eval(evidence: list[dict], ground_truth: dict, model: str = "claude-3-5-sonnet-latest") -> dict:
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return {"error": "ANTHROPIC_API_KEY not set"}

    prompt = (
        "Evaluate search quality for an investment research agent using the provided ground truth. "
        "For each provider/query, score 1-5: relevance, financial_signal, evidence_quality, noise_level (5=very noisy), and truth_alignment."
        "\n\nGROUND_TRUTH:\n"
        + json.dumps(ground_truth, ensure_ascii=False)
        + "\n\nDATA:\n"
        + json.dumps(evidence, ensure_ascii=False)
    )
    body = {
        "model": model,
        "max_tokens": 1800,
        "temperature": 0,
        "messages": [{"role": "user", "content": prompt}],
    }

    try:
        resp = httpx.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "x-api-key": key,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json=body,
            timeout=45,
        )
    except httpx.TimeoutException as e:
        return {"error": f"Anthropic timeout: {e}"}
    except Exception as e:
        return {"error": f"Anthropic request failed: {e}"}

    if resp.status_code != 200:
        return {"error": f"Anthropic HTTP {resp.status_code}", "body": resp.text[:500]}

    content = resp.json().get("content", [])
    text = content[0].get("text", "") if content else ""
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return {"raw": text}

In [25]:
# 0.4b Extract search outputs + LLM quality assessment (OpenAI or Anthropic)
import json
import re
from pathlib import Path


def _parse_llm_json(text: str) -> dict:
    raw = (text or "").strip()
    m = re.match(r"^```(?:json)?\s*(.*?)\s*```$", raw, flags=re.DOTALL | re.IGNORECASE)
    if m:
        raw = m.group(1).strip()

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        start = raw.find("{")
        end = raw.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(raw[start:end + 1])
        raise


def flatten_search_outputs(search_results: list[dict], provider: str) -> list[dict]:
    rows = []
    for item in search_results:
        query = item.get("query", "")
        snippets = item.get("snippets", []) or []
        normalized_snippets = []
        for s in snippets[:2]:
            normalized_snippets.append({
                "title": s.get("title", ""),
                "text": (s.get("description") or s.get("content") or "")[:180],
            })

        rows.append({
            "provider": provider,
            "query": query,
            "ok": bool(item.get("ok")),
            "latency_ms": round(float(item.get("latency_ms", 0)), 2),
            "num_results": int(item.get("num_results", 0) or 0),
            "answer": (item.get("answer") or "")[:350],
            "snippets": normalized_snippets,
        })
    return rows


def _chunked(items: list[dict], size: int) -> list[list[dict]]:
    return [items[i:i + size] for i in range(0, len(items), size)]


def _build_provider_summary(evaluations: list[dict]) -> list[dict]:
    grouped = {}
    for row in evaluations:
        provider = row.get("provider", "unknown")
        grouped.setdefault(provider, []).append(row)

    summary = []
    for provider, rows in grouped.items():
        n = max(len(rows), 1)
        avg_overall = sum(float(r.get("overall_quality", 0)) for r in rows) / n
        avg_truth = sum(float(r.get("truth_alignment", 0)) for r in rows) / n
        avg_noise = sum(float(r.get("noise_level", 0)) for r in rows) / n
        recommendation = "keep as primary" if avg_overall >= 4.0 else "use as fallback"
        summary.append({
            "provider": provider,
            "avg_overall_quality": round(avg_overall, 2),
            "avg_truth_alignment": round(avg_truth, 2),
            "avg_noise": round(avg_noise, 2),
            "recommendation": recommendation,
        })
    return sorted(summary, key=lambda x: x["avg_overall_quality"], reverse=True)


search_evidence = (
    flatten_search_outputs(brave_results, "brave_search")
    + flatten_search_outputs(brave_answers_results, "brave_answers")
    + flatten_search_outputs(tavily_results, "tavily")
)
ground_truth_payload = {"questions": GROUND_TRUTH_QA}

export_path = PROJECT_ROOT / "data" / "phase0_search_outputs.json"
export_path.parent.mkdir(parents=True, exist_ok=True)
export_path.write_text(json.dumps(search_evidence, indent=2), encoding="utf-8")

print(f"Extracted {len(search_evidence)} provider-query records")
print(f"Saved raw output to: {export_path}")
print(pd.DataFrame(search_evidence)[["provider", "query", "ok", "latency_ms", "num_results"]].to_string(index=False))


def call_openai_quality_eval(
    evidence: list[dict],
    ground_truth: dict,
    model: str = "gpt-4o-mini",
    timeout_s: int = 90,
    max_retries: int = 2,
) -> dict:
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        return {"error": "OPENAI_API_KEY not set"}

    prompt = (
        "You are evaluating search quality for an investment research agent. "
        "Use the provided ground truth to score each provider/query from 1-5 for: relevance, financial_signal, evidence_quality, noise_level (5 means very noisy), and truth_alignment (how well results align with ground truth key facts). "
        "Compute overall_quality as rounded average of relevance, financial_signal, evidence_quality, truth_alignment, and (6-noise_level). "
        "Return JSON only with this schema: "
        "{\"evaluations\": [{\"provider\": str, \"query\": str, \"relevance\": int, \"financial_signal\": int, \"evidence_quality\": int, \"noise_level\": int, \"truth_alignment\": int, \"overall_quality\": int, \"notes\": str}], \"provider_summary\": [{\"provider\": str, \"avg_overall_quality\": float, \"avg_truth_alignment\": float, \"avg_noise\": float, \"recommendation\": str}]}"
    )

    body = {
        "model": model,
        "temperature": 0,
        "messages": [
            {"role": "system", "content": "Respond with valid JSON only."},
            {
                "role": "user",
                "content": (
                    prompt
                    + "\n\nGROUND_TRUTH:\n"
                    + json.dumps(ground_truth, ensure_ascii=False)
                    + "\n\nDATA:\n"
                    + json.dumps(evidence, ensure_ascii=False)
                ),
            },
        ],
    }

    last_error = "unknown"
    for attempt in range(max_retries + 1):
        try:
            resp = httpx.post(
                "https://api.openai.com/v1/chat/completions",
                headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
                json=body,
                timeout=timeout_s,
            )
        except httpx.TimeoutException as e:
            last_error = f"OpenAI timeout: {e}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}
        except Exception as e:
            last_error = f"OpenAI request failed: {e}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}

        if resp.status_code in (408, 429, 500, 502, 503, 504):
            last_error = f"OpenAI HTTP {resp.status_code}: {resp.text[:200]}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}

        if resp.status_code != 200:
            return {"error": f"OpenAI HTTP {resp.status_code}", "body": resp.text[:500]}

        text = resp.json()["choices"][0]["message"]["content"]
        try:
            return _parse_llm_json(text)
        except Exception:
            return {"raw": text}

    return {"error": last_error}


def call_anthropic_quality_eval(
    evidence: list[dict],
    ground_truth: dict,
    model: str = "claude-3-5-sonnet-latest",
    timeout_s: int = 90,
    max_retries: int = 2,
) -> dict:
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return {"error": "ANTHROPIC_API_KEY not set"}

    prompt = (
        "Evaluate search quality for an investment research agent using the provided ground truth. "
        "For each provider/query, score 1-5: relevance, financial_signal, evidence_quality, noise_level (5=very noisy), and truth_alignment. "
        "Compute overall_quality as rounded average of relevance, financial_signal, evidence_quality, truth_alignment, and (6-noise_level). "
        "Return JSON only using schema: "
        "{\"evaluations\": [{\"provider\": str, \"query\": str, \"relevance\": int, \"financial_signal\": int, \"evidence_quality\": int, \"noise_level\": int, \"truth_alignment\": int, \"overall_quality\": int, \"notes\": str}], \"provider_summary\": [{\"provider\": str, \"avg_overall_quality\": float, \"avg_truth_alignment\": float, \"avg_noise\": float, \"recommendation\": str}]}"
        "\n\nGROUND_TRUTH:\n"
        + json.dumps(ground_truth, ensure_ascii=False)
        + "\n\nDATA:\n"
        + json.dumps(evidence, ensure_ascii=False)
    )

    body = {
        "model": model,
        "max_tokens": 1800,
        "temperature": 0,
        "messages": [{"role": "user", "content": prompt}],
    }

    last_error = "unknown"
    for attempt in range(max_retries + 1):
        try:
            resp = httpx.post(
                "https://api.anthropic.com/v1/messages",
                headers={
                    "x-api-key": key,
                    "anthropic-version": "2023-06-01",
                    "content-type": "application/json",
                },
                json=body,
                timeout=timeout_s,
            )
        except httpx.TimeoutException as e:
            last_error = f"Anthropic timeout: {e}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}
        except Exception as e:
            last_error = f"Anthropic request failed: {e}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}

        if resp.status_code in (408, 429, 500, 502, 503, 504):
            last_error = f"Anthropic HTTP {resp.status_code}: {resp.text[:200]}"
            if attempt < max_retries:
                time.sleep(1.5 * (attempt + 1))
                continue
            return {"error": last_error}

        if resp.status_code != 200:
            return {"error": f"Anthropic HTTP {resp.status_code}", "body": resp.text[:500]}

        content = resp.json().get("content", [])
        text = content[0].get("text", "") if content else ""
        try:
            return _parse_llm_json(text)
        except Exception:
            return {"raw": text}

    return {"error": last_error}


# Choose one: "openai" or "anthropic"
EVAL_PROVIDER = "openai"
RUN_LLM_EVAL = True
EVAL_BATCH_SIZE = 6
EVAL_TIMEOUT_SECONDS = 90
EVAL_MAX_RETRIES = 2

quality_report = {}
if RUN_LLM_EVAL:
    batches = _chunked(search_evidence, EVAL_BATCH_SIZE)
    all_evaluations = []

    for i, batch in enumerate(batches, start=1):
        print(f"Evaluating batch {i}/{len(batches)} ({len(batch)} records)...")
        if EVAL_PROVIDER == "openai":
            batch_report = call_openai_quality_eval(
                batch,
                ground_truth_payload,
                timeout_s=EVAL_TIMEOUT_SECONDS,
                max_retries=EVAL_MAX_RETRIES,
            )
        elif EVAL_PROVIDER == "anthropic":
            batch_report = call_anthropic_quality_eval(
                batch,
                ground_truth_payload,
                timeout_s=EVAL_TIMEOUT_SECONDS,
                max_retries=EVAL_MAX_RETRIES,
            )
        else:
            batch_report = {"error": f"Unknown EVAL_PROVIDER: {EVAL_PROVIDER}"}

        if batch_report.get("error"):
            quality_report = {"error": f"Batch {i} failed: {batch_report['error']}"}
            break

        batch_evals = batch_report.get("evaluations", [])
        if not batch_evals and batch_report.get("raw"):
            quality_report = {"error": f"Batch {i} returned unparsable payload", "raw": batch_report.get("raw", "")[:1000]}
            break

        all_evaluations.extend(batch_evals)

    if not quality_report.get("error"):
        quality_report = {
            "evaluations": all_evaluations,
            "provider_summary": _build_provider_summary(all_evaluations),
        }

if quality_report.get("error"):
    print(f"[FAIL] Quality eval: {quality_report['error']}")
    if quality_report.get("raw"):
        print("\nRaw snippet:")
        print(str(quality_report["raw"])[:1000])
elif quality_report.get("evaluations"):
    eval_df = pd.DataFrame(quality_report["evaluations"])
    print("\nPer query/provider quality scores:")
    show_cols = [
        c for c in [
            "provider",
            "query",
            "relevance",
            "financial_signal",
            "evidence_quality",
            "truth_alignment",
            "noise_level",
            "overall_quality",
        ] if c in eval_df.columns
    ]
    print(eval_df[show_cols].to_string(index=False))

    if quality_report.get("provider_summary"):
        print("\nProvider summary:")
        print(pd.DataFrame(quality_report["provider_summary"]).to_string(index=False))

    eval_path = PROJECT_ROOT / "data" / "phase0_search_quality_eval.json"
    eval_path.write_text(json.dumps(quality_report, indent=2), encoding="utf-8")
    print(f"\nSaved quality evaluation to: {eval_path}")
else:
    print("[WARN] Unexpected response format, showing raw output")
    print(json.dumps(quality_report, indent=2)[:2000])

Extracted 18 provider-query records
Saved raw output to: /home/deploy_invest_ai/investment-agent/data/phase0_search_outputs.json
     provider                                                                      query   ok  latency_ms  num_results
 brave_search AAPL FY2025 Q4: total revenue and YoY growth from official company release True      954.17            5
 brave_search                            AAPL FY2025 Q4: Services revenue and YoY growth True      927.95            5
 brave_search                  Apple SEC CIK and exact data.sec.gov submissions endpoint True      977.46            5
 brave_search                  SEC annual vs quarterly filing forms for public companies True      966.05            5
 brave_search                        SEC insider ownership reporting forms and deadlines True      958.71            5
 brave_search                           SEC form for material events and filing deadline True      957.58            5
brave_answers AAPL FY2025 Q4: total re

---
## 0.4d Side-by-side Visualization

Compare provider quality scores side by side with plain tables (no plotting dependencies).

In [30]:
# 0.4d side-by-side table view (pure pandas, no matplotlib)

if "quality_report" not in globals() or not quality_report.get("evaluations"):
    eval_path = PROJECT_ROOT / "data" / "phase0_search_quality_eval.json"
    if eval_path.exists():
        quality_report = json.loads(eval_path.read_text(encoding="utf-8"))
    else:
        raise ValueError("Run 0.4b first (or ensure phase0_search_quality_eval.json exists)")

viz_df = pd.DataFrame(quality_report.get("evaluations", []))
if viz_df.empty:
    raise ValueError("No evaluations found in quality_report")

metric_cols = [
    c for c in [
        "overall_quality",
        "truth_alignment",
        "relevance",
        "financial_signal",
        "evidence_quality",
        "noise_level",
    ]
    if c in viz_df.columns
]

# Wide table: one row per query with provider metrics side-by-side.
wide_rows = []
for q in sorted(viz_df["query"].unique().tolist()):
    row = {"query": q}
    for provider in sorted(viz_df["provider"].unique().tolist()):
        sub = viz_df[(viz_df["query"] == q) & (viz_df["provider"] == provider)]
        if sub.empty:
            continue
        rec = sub.iloc[0]
        for metric in metric_cols:
            row[f"{provider}__{metric}"] = rec.get(metric)
    wide_rows.append(row)

wide_df = pd.DataFrame(wide_rows)
if len(wide_df.columns) > 1:
    numeric_cols = [c for c in wide_df.columns if c != "query"]
    wide_df[numeric_cols] = wide_df[numeric_cols].apply(pd.to_numeric, errors="coerce").round(2)

print("Side-by-side quality table:")
display(wide_df)

# Compact provider averages table
provider_avg = (
    viz_df.groupby("provider", as_index=False)[metric_cols]
    .mean(numeric_only=True)
    .sort_values("overall_quality", ascending=False)
    .round(2)
)
print("Provider average scores:")
display(provider_avg)

# Query x provider matrix (overall quality)
heat = (
    viz_df.pivot_table(index="query", columns="provider", values="overall_quality", aggfunc="mean")
    .round(2)
)
print("Overall quality matrix (query x provider):")
display(heat)

Side-by-side quality table:


,query,brave_answers__overall_quality,brave_answers__truth_alignment,brave_answers__relevance,brave_answers__financial_signal,brave_answers__evidence_quality,brave_answers__noise_level,brave_search__overall_quality,brave_search__truth_alignment,brave_search__relevance,brave_search__financial_signal,brave_search__evidence_quality,brave_search__noise_level,tavily__overall_quality,tavily__truth_alignment,tavily__relevance,tavily__financial_signal,tavily__evidence_quality,tavily__noise_level
0,AAPL FY2025 Q4: Services revenue and YoY growth,1,1,1,1,1,5,5,5,5,5,4,2,4,4,4,5,4,3
1,AAPL FY2025 Q4: total revenue and YoY growth from official company release,1,1,1,1,1,5,5,5,5,5,4,2,5,5,5,5,5,2
2,Apple SEC CIK and exact data.sec.gov submissions endpoint,3,4,3,3,3,3,3,4,4,3,3,3,4,5,5,5,3,4
3,SEC annual vs quarterly filing forms for public companies,4,5,4,4,4,2,3,4,4,2,3,2,4,5,5,4,4,3
4,SEC form for material events and filing deadline,4,5,4,4,4,2,3,4,4,2,3,2,4,5,5,4,4,3
5,SEC insider ownership reporting forms and deadlines,4,5,4,4,4,2,3,4,4,2,3,2,4,5,5,4,4,3


Provider average scores:


,provider,overall_quality,truth_alignment,relevance,financial_signal,evidence_quality,noise_level
2,tavily,4.17,4.83,4.83,4.50,4.00,3.00
1,brave_search,3.67,4.33,4.33,3.17,3.33,2.17
0,brave_answers,2.83,3.50,2.83,2.83,2.83,3.17


Overall quality matrix (query x provider):


provider,brave_answers,brave_search,tavily
query,,,
AAPL FY2025 Q4: Services revenue and YoY growth,1.0,5.0,4.0
AAPL FY2025 Q4: total revenue and YoY growth from official company release,1.0,5.0,5.0
Apple SEC CIK and exact data.sec.gov submissions endpoint,3.0,3.0,4.0
SEC annual vs quarterly filing forms for public companies,4.0,3.0,4.0
SEC form for material events and filing deadline,4.0,3.0,4.0
SEC insider ownership reporting forms and deadlines,4.0,3.0,4.0


---
## 0.4e Hard Question Re-test

Run a tougher benchmark set (multi-step regulatory reasoning + filing mapping) and score providers again before final recommendation.

In [ ]:
HARD_SAMPLE_QUERIES = [
    "Build a filing timeline after quarter-end: include 10-Q, 10-K, and 8-K timing.",
    "What SEC forms disclose insider ownership changes and what is the typical Form 4 deadline?",
    "How do you construct the canonical SEC submissions URL from CIK, using Apple as example?",
    "Distinguish Schedule 13D vs Form 4 in one sentence each (purpose and filer).",
    "Which filing is best for annual audited financials/risk factors vs current material events?",
    "Event mapping: CFO resigns and an insider buys shares the same week; list required SEC forms.",
]

HARD_GROUND_TRUTH_QA = {
    HARD_SAMPLE_QUERIES[0]: {
        "model_answer": "10-Q is quarterly reporting, 10-K is annual reporting, and 8-K is current material-event reporting (typically within 4 business days for specified items).",
        "key_facts": ["10-Q quarterly", "10-K annual", "8-K current report", "8-K 4 business days"],
        "sources": ["https://www.sec.gov/forms"],
    },
    HARD_SAMPLE_QUERIES[1]: {
        "model_answer": "Insider ownership disclosures use Forms 3, 4, and 5. Form 4 is typically due within 2 business days of a change in beneficial ownership.",
        "key_facts": ["Forms 3/4/5", "Form 4 2 business days"],
        "sources": [
            "https://www.sec.gov/forms",
            "https://www.investor.gov/introduction-investing/investing-basics/glossary/form-4",
        ],
    },
    HARD_SAMPLE_QUERIES[2]: {
        "model_answer": "Pad CIK to 10 digits and use https://data.sec.gov/submissions/CIK{CIK10}.json. Apple example: https://data.sec.gov/submissions/CIK0000320193.json",
        "key_facts": ["CIK padded to 10", "CIK0000320193.json"],
        "sources": [
            "https://www.sec.gov/files/company_tickers.json",
            "https://data.sec.gov/submissions/CIK0000320193.json",
        ],
    },
    HARD_SAMPLE_QUERIES[3]: {
        "model_answer": "Schedule 13D is for beneficial owners crossing 5% ownership; Form 4 is for insiders (officers/directors/10% holders) reporting ownership changes.",
        "key_facts": ["13D >5% beneficial ownership", "Form 4 insiders ownership changes"],
        "sources": ["https://www.sec.gov/forms"],
    },
    HARD_SAMPLE_QUERIES[4]: {
        "model_answer": "Use 10-K for annual audited financial statements and risk factors; use 8-K for current material events.",
        "key_facts": ["10-K audited annual financials", "10-K risk factors", "8-K material events"],
        "sources": ["https://www.sec.gov/forms"],
    },
    HARD_SAMPLE_QUERIES[5]: {
        "model_answer": "CFO resignation is reported on Form 8-K; insider share purchase is reported on Form 4.",
        "key_facts": ["CFO resignation -> 8-K", "Insider purchase -> Form 4"],
        "sources": ["https://www.sec.gov/forms"],
    },
}


def run_provider_suite(queries: list[str]) -> tuple[list[dict], list[dict], list[dict]]:
    out_brave_search = []
    out_brave_answers = []
    out_tavily = []

    for q in queries:
        out_brave_search.append({"query": q, **call_brave_search(q)})
        out_brave_answers.append({"query": q, **call_brave_answers(q)})
        out_tavily.append({"query": q, **call_tavily_search(q)})

    return out_brave_search, out_brave_answers, out_tavily


hard_brave_results, hard_brave_answers_results, hard_tavily_results = run_provider_suite(HARD_SAMPLE_QUERIES)

hard_rows = []
for i, q in enumerate(HARD_SAMPLE_QUERIES):
    bs = hard_brave_results[i]
    ba = hard_brave_answers_results[i]
    tv = hard_tavily_results[i]
    hard_rows.append({
        "query": q,
        "brave_search_ok": bs.get("ok", False),
        "brave_search_latency_ms": round(float(bs.get("latency_ms", 0)), 2),
        "brave_answers_ok": ba.get("ok", False),
        "brave_answers_latency_ms": round(float(ba.get("latency_ms", 0)), 2),
        "tavily_ok": tv.get("ok", False),
        "tavily_latency_ms": round(float(tv.get("latency_ms", 0)), 2),
    })

hard_df_abc = pd.DataFrame(hard_rows)
print("Hard benchmark provider run table:")
display(hard_df_abc)

hard_search_evidence = (
    flatten_search_outputs(hard_brave_results, "brave_search")
    + flatten_search_outputs(hard_brave_answers_results, "brave_answers")
    + flatten_search_outputs(hard_tavily_results, "tavily")
)
hard_ground_truth_payload = {"questions": HARD_GROUND_TRUTH_QA}

hard_quality_report = {}
if RUN_LLM_EVAL:
    batches = _chunked(hard_search_evidence, EVAL_BATCH_SIZE)
    all_evaluations = []

    for i, batch in enumerate(batches, start=1):
        print(f"Hard benchmark eval batch {i}/{len(batches)} ({len(batch)} records)...")
        if EVAL_PROVIDER == "openai":
            batch_report = call_openai_quality_eval(
                batch,
                hard_ground_truth_payload,
                timeout_s=EVAL_TIMEOUT_SECONDS,
                max_retries=EVAL_MAX_RETRIES,
            )
        elif EVAL_PROVIDER == "anthropic":
            batch_report = call_anthropic_quality_eval(
                batch,
                hard_ground_truth_payload,
                timeout_s=EVAL_TIMEOUT_SECONDS,
                max_retries=EVAL_MAX_RETRIES,
            )
        else:
            batch_report = {"error": f"Unknown EVAL_PROVIDER: {EVAL_PROVIDER}"}

        if batch_report.get("error"):
            hard_quality_report = {"error": f"Hard benchmark batch {i} failed: {batch_report['error']}"}
            break

        batch_evals = batch_report.get("evaluations", [])
        if not batch_evals and batch_report.get("raw"):
            hard_quality_report = {
                "error": f"Hard benchmark batch {i} unparsable payload",
                "raw": str(batch_report.get("raw", ""))[:1000],
            }
            break

        all_evaluations.extend(batch_evals)

    if not hard_quality_report.get("error"):
        hard_quality_report = {
            "evaluations": all_evaluations,
            "provider_summary": _build_provider_summary(all_evaluations),
        }

if hard_quality_report.get("error"):
    print(f"[FAIL] Hard benchmark quality eval: {hard_quality_report['error']}")
    if hard_quality_report.get("raw"):
        print("\nRaw snippet:")
        print(hard_quality_report["raw"])
elif hard_quality_report.get("evaluations"):
    hard_eval_df = pd.DataFrame(hard_quality_report["evaluations"])
    print("\nHard benchmark: per query/provider quality scores")
    hard_show_cols = [
        c for c in [
            "provider",
            "query",
            "relevance",
            "financial_signal",
            "evidence_quality",
            "truth_alignment",
            "noise_level",
            "overall_quality",
        ] if c in hard_eval_df.columns
    ]
    display(hard_eval_df[hard_show_cols])

    print("Hard benchmark: provider summary")
    display(pd.DataFrame(hard_quality_report.get("provider_summary", [])))

    hard_eval_path = PROJECT_ROOT / "data" / "phase0_search_quality_eval_hard.json"
    hard_eval_path.write_text(json.dumps(hard_quality_report, indent=2), encoding="utf-8")
    print(f"Saved hard benchmark quality evaluation to: {hard_eval_path}")

---
## 0.5 SEC Ground-Truth Validation

Validate that SEC EDGAR endpoints return the canonical identifiers and filing metadata used as factual anchors in this notebook's benchmarks.

In [31]:
from collections import Counter

TICKER_TO_TEST = "AAPL"
EXPECTED_CIK = 320193

def sec_get_tickers() -> dict:
    url = "https://www.sec.gov/files/company_tickers.json"
    headers = {"User-Agent": "InvestmentAgent research@example.com"}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, headers=headers, timeout=15)
        latency = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency}
        return {"ok": True, "data": resp.json(), "latency_ms": latency}
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

def sec_get_submissions(cik: int) -> dict:
    cik_padded = str(cik).zfill(10)
    url = f"https://data.sec.gov/submissions/CIK{cik_padded}.json"
    headers = {"User-Agent": "InvestmentAgent research@example.com"}
    t0 = time.perf_counter()
    try:
        resp = httpx.get(url, headers=headers, timeout=15)
        latency = (time.perf_counter() - t0) * 1000
        if resp.status_code != 200:
            return {"error": f"HTTP {resp.status_code}", "latency_ms": latency}
        data = resp.json()
        recent = data.get("filings", {}).get("recent", {})
        forms = recent.get("form", []) or []
        accession = recent.get("accessionNumber", []) or []
        filing_dates = recent.get("filingDate", []) or []
        preview = list(zip(forms[:5], filing_dates[:5], accession[:5]))
        return {
            "ok": True,
            "name": data.get("name", ""),
            "forms": forms,
            "preview": preview,
            "latency_ms": latency,
        }
    except Exception as e:
        return {"error": str(e), "latency_ms": (time.perf_counter() - t0) * 1000}

tickers_r = sec_get_tickers()
sub_r = {}
cik = None

if not tickers_r.get("ok"):
    print(f"[FAIL] Tickers endpoint: {tickers_r.get('error')}")
else:
    print(f"[PASS] Tickers endpoint: {tickers_r['latency_ms']:.0f}ms")
    for row in tickers_r["data"].values():
        if row.get("ticker") == TICKER_TO_TEST:
            cik = int(row.get("cik_str"))
            break

if cik is not None:
    sub_r = sec_get_submissions(cik)
    if sub_r.get("ok"):
        print(f"[PASS] Submissions endpoint: {sub_r['latency_ms']:.0f}ms")
        print(f"  Company: {sub_r.get('name', '')}")
        print(f"  First 5 recent filings (form, date, accession): {sub_r.get('preview', [])}")
        form_counts = Counter(sub_r.get("forms", []))
        print(f"  Top forms: {form_counts.most_common(5)}")
    else:
        print(f"[FAIL] Submissions endpoint: {sub_r.get('error')}")

checks = {
    "resolved_cik_matches_expected": cik == EXPECTED_CIK,
    "submissions_endpoint_ok": bool(sub_r.get("ok")),
    "company_name_contains_apple": "apple" in (sub_r.get("name", "").lower() if sub_r else ""),
    "has_recent_forms": len(sub_r.get("forms", [])) > 0 if sub_r else False,
}

print("\nSEC validation checks:")
for name, ok in checks.items():
    print(f" - {name}: {'PASS' if ok else 'FAIL'}")

sec_pass = all(checks.values())
print(f"\nSection 0.5: {'PASS' if sec_pass else 'FAIL'}")

[PASS] Tickers: 229ms
[PASS] Submissions for AAPL (CIK 320193): 357ms
  Recent forms: []

Section 0.5: PASS


---
## 0.6 Deployment Decision Table

Combine latency, cost, and quality (base + hard benchmark) into a practical provider routing recommendation.

In [32]:
def _summary_lookup(report: dict, provider: str, key: str) -> float:
    rows = report.get("provider_summary", []) if isinstance(report, dict) else []
    for r in rows:
        if r.get("provider") == provider:
            return float(r.get(key, 0))
    return 0.0

base_report = quality_report if isinstance(quality_report, dict) else {}
hard_report = hard_quality_report if isinstance(globals().get("hard_quality_report", {}), dict) else {}

sec_latency = sub_r.get("latency_ms", 0) if (cik and sub_r.get("ok")) else 0

decision_rows = [
    {
        "provider": "brave_search",
        "cost_estimate": "~$0.005/call",
        "monthly_limit": "2000",
        "avg_latency_ms": float(df_abc["brave_search_latency_ms"].mean()) if "df_abc" in globals() else 0,
    },
    {
        "provider": "brave_answers",
        "cost_estimate": "~$0.004/call + tokens",
        "monthly_limit": "2000",
        "avg_latency_ms": float(df_abc["brave_answers_latency_ms"].mean()) if "df_abc" in globals() else 0,
    },
    {
        "provider": "tavily",
        "cost_estimate": "~$0.008/call",
        "monthly_limit": "1000",
        "avg_latency_ms": float(df_abc["tavily_latency_ms"].mean()) if "df_abc" in globals() else 0,
    },
]

for row in decision_rows:
    p = row["provider"]
    row["base_overall_quality"] = _summary_lookup(base_report, p, "avg_overall_quality")
    row["hard_overall_quality"] = _summary_lookup(hard_report, p, "avg_overall_quality")
    row["hard_truth_alignment"] = _summary_lookup(hard_report, p, "avg_truth_alignment")

    quality_anchor = row["hard_overall_quality"] if row["hard_overall_quality"] > 0 else row["base_overall_quality"]
    row["decision_score"] = round((quality_anchor * 0.75) + (5.0 * (1.0 / max(row["avg_latency_ms"], 1)) * 1000 * 0.25), 3)

decision_df = pd.DataFrame(decision_rows).sort_values("decision_score", ascending=False)
for col in ["avg_latency_ms", "base_overall_quality", "hard_overall_quality", "hard_truth_alignment"]:
    decision_df[col] = decision_df[col].round(2)

print("Provider decision table (higher decision_score is better):")
display(decision_df)

if not decision_df.empty:
    primary = decision_df.iloc[0]["provider"]
    fallbacks = decision_df.iloc[1:]["provider"].tolist()
    print(f"\nRecommended routing: primary={primary}, fallback_order={fallbacks}")

print(f"SEC EDGAR latency (ground-truth anchor source): {sec_latency:.0f}ms")
print("\nSection 0.6: PASS")

          API                  cost      limit  avg_latency_ms
 Brave Search          ~$0.005/call 2000/month      956.989065
Brave Answers ~$0.004/call + tokens 2000/month     7834.001440
Tavily Search          ~$0.008/call 1000/month      672.002802
    SEC EDGAR                  Free   10 req/s      357.025895

Config: max_calls_per_member_per_cycle strategy=20, skeptic=8, risk=7; max_total=35

Section 0.6: PASS


---
## 0.7 Routing Policy Dry-Run

Simulate provider routing with budget and fallback using the decision table from 0.6, so the outcome maps directly to production configuration choices.

In [33]:
if "decision_df" not in globals() or decision_df.empty:
    raise ValueError("Run 0.6 first to build decision_df")

provider_order = decision_df["provider"].tolist()
provider_budget = {"brave_search": 8, "brave_answers": 4, "tavily": 6}
provider_used = {k: 0 for k in provider_budget}

# Simulated live request stream (mixed easy + hard prompts).
request_stream = [
    {"member": "strategy", "query": SAMPLE_QUERIES[0], "difficulty": "easy"},
    {"member": "strategy", "query": HARD_SAMPLE_QUERIES[0], "difficulty": "hard"},
    {"member": "skeptic", "query": HARD_SAMPLE_QUERIES[3], "difficulty": "hard"},
    {"member": "strategy", "query": SAMPLE_QUERIES[2], "difficulty": "easy"},
    {"member": "risk", "query": HARD_SAMPLE_QUERIES[5], "difficulty": "hard"},
]

routing_log = []
for req in request_stream:
    picked = None
    for provider in provider_order:
        if provider_used.get(provider, 0) < provider_budget.get(provider, 0):
            picked = provider
            break

    if picked is None:
        routing_log.append({**req, "provider": None, "status": "budget_exhausted"})
        print(f"[SKIP] No provider budget left for query: {req['query'][:60]}...")
        continue

    provider_used[picked] += 1
    routing_log.append({**req, "provider": picked, "status": "assigned"})
    print(
        f"[ROUTE] {req['member']} {req['difficulty']} -> {picked} "
        f"({provider_used[picked]}/{provider_budget[picked]})"
    )

routing_df = pd.DataFrame(routing_log)
print("\nRouting dry-run output:")
display(routing_df)

policy = {
    "primary_provider": provider_order[0],
    "fallback_order": provider_order[1:],
    "provider_budget": provider_budget,
    "quality_min_threshold": 3.5,
    "timeout_seconds": EVAL_TIMEOUT_SECONDS if "EVAL_TIMEOUT_SECONDS" in globals() else 90,
}

print("\nProposed runtime policy (for config translation):")
print(json.dumps(policy, indent=2))
print("\nSection 0.7: PASS (routing policy dry-run complete)")

[CALL] strategy web_search (1/20, total 1/35)
[CALL] strategy news_search (2/20, total 2/35)
[CALL] skeptic web_search (1/8, total 3/35)

Cache keys: ['AAPL|web_search|aapl earnings', 'AAPL|news_search|aapl analyst', 'AAPL|web_search|aapl risks']

Section 0.7: PASS (cache + budget logic validated)


---
## 0.8 End-to-End Single-Ticker Walkthrough (CB)

Trace one ticker through the committee pipeline (strategy -> skeptic -> risk), show where web search is allowed, enforce follow-up budgets, and compare static vs web-enriched decisions.

In [ ]:
# 0.8 CB end-to-end walkthrough: static vs web-enriched decisions
from src.utils.config import get_settings

settings = get_settings()
CB_TICKER = "CB"
CB_INSTRUMENT = "CB_US_EQ"

# Stage policy from current code/config
stage_policy = pd.DataFrame([
    {
        "stage": "strategy (Claude)",
        "role": "opportunity synthesis",
        "web_search_enabled_config": bool(settings.research_enabled and settings.strategy_research_enabled),
        "impl_tool_loop_limit": 8,   # src/agents/strategy/engine.py max_iterations
        "suggested_followup_queries": 4,
        "hard_cap_calls_per_cycle": settings.research_max_calls_per_member_per_cycle.get("strategy", 20),
        "notes": "Tools: web_search/news_search/sector_search/sec_search; 30s total loop timeout",
    },
    {
        "stage": "moderation skeptic (GPT-4o)",
        "role": "bear-case challenge",
        "web_search_enabled_config": bool(settings.research_enabled and settings.skeptic_research_enabled),
        "impl_tool_loop_limit": 4,   # src/agents/moderation/openai_mod.py max_iter
        "suggested_followup_queries": 2,
        "hard_cap_calls_per_cycle": settings.research_max_calls_per_member_per_cycle.get("skeptic", 8),
        "notes": "Prompt says use sparingly (1-2 searches)",
    },
    {
        "stage": "moderation risk (Gemini)",
        "role": "growth/risk/confidence scoring",
        "web_search_enabled_config": bool(settings.research_enabled and settings.risk_research_enabled),
        "impl_tool_loop_limit": 0,   # gemini_mod notes tool-use loop TBD
        "suggested_followup_queries": 0,
        "hard_cap_calls_per_cycle": settings.research_max_calls_per_member_per_cycle.get("risk", 7),
        "notes": "Current implementation is single-turn; no active tool-use loop",
    },
])

print("Pipeline web-search policy (actual implementation):")
display(stage_policy)
print(f"Global research cap per cycle: {settings.research_max_total_calls_per_cycle}")


def cb_query_plan(ticker: str) -> dict[str, list[str]]:
    return {
        "strategy": [
            f"{ticker} earnings guidance revisions 2026",
            f"{ticker} underwriting margin trend catastrophe exposure",
            f"{ticker} valuation vs peers insurance industry",
            f"{ticker} 8-K or 10-Q risk factor changes",
        ],
        "skeptic": [
            f"{ticker} analyst downgrade downside scenario",
            f"{ticker} litigation regulatory capital adequacy concerns",
        ],
        "risk": [],
    }


question_plan = cb_query_plan(CB_TICKER)
print("\nShaped follow-up question templates by stage:")
for stage, qs in question_plan.items():
    print(f"\n{stage}:")
    for i, q in enumerate(qs, start=1):
        print(f"  {i}. {q}")


# Fetch concise web evidence for CB using current best provider routing
provider_order = decision_df["provider"].tolist() if "decision_df" in globals() and not decision_df.empty else ["tavily", "brave_search", "brave_answers"]
primary_provider = provider_order[0]


def provider_search(provider: str, query: str, ticker: str) -> dict:
    if provider == "tavily":
        return call_tavily_search(query)
    if provider == "brave_search":
        return call_brave_search(query)
    if provider == "brave_answers":
        return call_brave_answers(query)
    return {"error": f"Unknown provider {provider}"}


def run_stage_search(stage: str, ticker: str, max_q: int) -> list[dict]:
    out = []
    for q in question_plan.get(stage, [])[:max_q]:
        res = provider_search(primary_provider, q, ticker)
        out.append({"stage": stage, "query": q, "provider": primary_provider, **res})
    return out


strategy_search = run_stage_search("strategy", CB_TICKER, 4)
skeptic_search = run_stage_search("skeptic", CB_TICKER, 2)
risk_search = run_stage_search("risk", CB_TICKER, 0)
cb_search_log = strategy_search + skeptic_search + risk_search

print(f"\nCB web evidence collected via {primary_provider}:")
display(pd.DataFrame([
    {
        "stage": r.get("stage"),
        "query": r.get("query"),
        "ok": r.get("ok", False),
        "latency_ms": round(float(r.get("latency_ms", 0)), 2),
        "num_results": r.get("num_results", 0),
        "answer_preview": (r.get("answer") or "")[:120],
    }
    for r in cb_search_log
]))


def _compact_evidence(rows: list[dict]) -> str:
    chunks = []
    for row in rows:
        snippets = row.get("snippets", []) or []
        top = snippets[0] if snippets else {}
        txt = (top.get("description") or top.get("content") or row.get("answer") or "")[:260]
        chunks.append(f"- [{row.get('stage')}] {row.get('query')} => {txt}")
    return "\n".join(chunks)


cb_static_context = (
    f"Ticker: {CB_INSTRUMENT}. "
    "Use only internal baseline assumptions (no web evidence). "
    "Return conservative judgement with clear uncertainty when data is missing."
)
cb_web_context = _compact_evidence(cb_search_log)


def _parse_json_response(text: str) -> dict:
    try:
        return _parse_llm_json(text)
    except Exception:
        return {"raw": text[:1500]}


def claude_decision(context_blob: str, mode_label: str) -> dict:
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        return {"error": "ANTHROPIC_API_KEY not set"}
    prompt = (
        f"Role: Strategy (Claude). Mode={mode_label}.\n"
        f"Ticker: {CB_INSTRUMENT}.\n"
        "Decide BUY/HOLD/REDUCE/SELL with conviction 0-100 and short reasoning.\n"
        "Respond JSON: {action, conviction, reasoning, key_risks, key_catalysts}.\n\n"
        f"Context:\n{context_blob}"
    )
    body = {
        "model": "claude-3-5-sonnet-latest",
        "max_tokens": 500,
        "temperature": 0.2,
        "messages": [{"role": "user", "content": prompt}],
    }
    try:
        resp = httpx.post(
            "https://api.anthropic.com/v1/messages",
            headers={
                "x-api-key": key,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json=body,
            timeout=60,
        )
        if resp.status_code != 200:
            return {"error": f"Anthropic HTTP {resp.status_code}", "body": resp.text[:300]}
        txt = (resp.json().get("content", [{}])[0] or {}).get("text", "")
        return _parse_json_response(txt)
    except Exception as e:
        return {"error": str(e)}


def gpt_skeptic_decision(context_blob: str, mode_label: str) -> dict:
    key = os.environ.get("OPENAI_API_KEY")
    if not key:
        return {"error": "OPENAI_API_KEY not set"}
    prompt = (
        f"Role: Skeptic (GPT-4o). Mode={mode_label}.\n"
        f"Ticker: {CB_INSTRUMENT}.\n"
        "Challenge the thesis and output verdict AGREE/DISAGREE/MODIFY with confidence 1-10.\n"
        "Respond JSON: {verdict, confidence, reasoning, disconfirming_evidence}.\n\n"
        f"Context:\n{context_blob}"
    )
    body = {
        "model": "gpt-4o-mini",
        "temperature": 0.2,
        "messages": [
            {"role": "system", "content": "Return JSON only."},
            {"role": "user", "content": prompt},
        ],
    }
    try:
        resp = httpx.post(
            "https://api.openai.com/v1/chat/completions",
            headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
            json=body,
            timeout=60,
        )
        if resp.status_code != 200:
            return {"error": f"OpenAI HTTP {resp.status_code}", "body": resp.text[:300]}
        txt = resp.json()["choices"][0]["message"]["content"]
        return _parse_json_response(txt)
    except Exception as e:
        return {"error": str(e)}


def gemini_risk_decision(context_blob: str, mode_label: str) -> dict:
    # Gemini risk stage currently has no tool loop in code; this is a single-turn assessment.
    key = os.environ.get("GOOGLE_AI_API_KEY")
    if not key:
        return {"error": "GOOGLE_AI_API_KEY not set"}
    try:
        from google import genai
        from google.genai import types
    except Exception as e:
        return {"error": f"google-genai package unavailable: {e}"}

    prompt = (
        f"Role: Risk assessor (Gemini). Mode={mode_label}.\n"
        f"Ticker: {CB_INSTRUMENT}.\n"
        "Output JSON: {verdict, growth_score, risk_score, confidence_score, assessment}.\n\n"
        f"Context:\n{context_blob}"
    )
    try:
        client = genai.Client(api_key=key)
        resp = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
            config=types.GenerateContentConfig(max_output_tokens=500, temperature=0.2),
        )
        txt = resp.text or ""
        return _parse_json_response(txt)
    except Exception as e:
        return {"error": str(e)}


cb_decision_compare = {
    "claude_static": claude_decision(cb_static_context, "static"),
    "claude_web": claude_decision(cb_web_context, "web_enriched"),
    "gpt_static": gpt_skeptic_decision(cb_static_context, "static"),
    "gpt_web": gpt_skeptic_decision(cb_web_context, "web_enriched"),
    "gemini_static": gemini_risk_decision(cb_static_context, "static"),
    "gemini_web": gemini_risk_decision(cb_web_context, "web_enriched"),
}

print("\nStatic vs web-enriched committee outputs for CB:")
display(pd.DataFrame([
    {"model_stage": k, "output": json.dumps(v)[:800]} for k, v in cb_decision_compare.items()
]))

# Simple delta summary
summary_rows = []
for stage in ["claude", "gpt", "gemini"]:
    s = cb_decision_compare.get(f"{stage}_static", {})
    w = cb_decision_compare.get(f"{stage}_web", {})
    summary_rows.append({
        "stage": stage,
        "static_action_or_verdict": s.get("action") or s.get("verdict") or "n/a",
        "web_action_or_verdict": w.get("action") or w.get("verdict") or "n/a",
        "static_confidence": s.get("conviction") or s.get("confidence") or s.get("confidence_score") or "n/a",
        "web_confidence": w.get("conviction") or w.get("confidence") or w.get("confidence_score") or "n/a",
        "changed": (s.get("action") != w.get("action")) or (s.get("verdict") != w.get("verdict")),
    })

print("Decision delta summary (web-enriched vs static):")
display(pd.DataFrame(summary_rows))
print("\nSection 0.8: PASS (CB end-to-end walkthrough complete)")

---
## Phase 0 Complete

Document Brave vs Tavily recommendation in AGENTIC_RESEARCH.md. Proceed to Phase A (src/agents/research/).